In [52]:
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy.sql import text
import warnings
import sys
import os # Esto sube un nivel en las carpetas para encontrar la raíz del proyecto
from dotenv import load_dotenv


# Verificar dónde está buscando el .env
print("Directorio actual:", os.getcwd())

# Cargar y comprobar variables
load_dotenv("../.env")

Directorio actual: c:\Users\danmo\Desktop\ADALAB\proyecto_fuera_clase\proyecto_ecommerce\e-commerce-operations-insights\notebooks


True

In [53]:
warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None) # para poder visualizar todas las columnas de los DataFrames

In [54]:
df_data = pd.read_csv("../files/processed/dataco_supply_chain.csv", sep=None, engine="python", encoding="utf-8")

In [55]:
df_data.sample(5)

,type,days_for_shipping_real,days_for_shipment_scheduled,benefit_per_order,sales_per_customer,delivery_status,late_delivery_risk,category_id,category_name,customer_city,customer_country,customer_fname,customer_id,customer_lname,customer_segment,customer_state,customer_street,customer_zipcode,department_id,department_name,latitude,longitude,order_city,order_country,order_customer_id,order_id,order_item_cardprod_id,order_item_discount,order_item_discount_rate,order_item_id,order_item_product_price,order_item_profit_ratio,order_item_quantity,sales,order_item_total,order_profit_per_order,order_region,order_status,product_card_id,product_category_id,product_name,product_price,shipping_mode,delivery_date,order_date,order_time
83533,DEBIT,3,4,39.00,129.99,Advance shipping,0,18,Men's Footwear,New York,EE. UU.,Tyler,827,Smith,Consumer,NY,5505 Grand Cloud Farm,10031,4,Apparel,40.790771,-74.008522,Teresina,Brasil,827,55297,403,0.00,0.00,138290,129.99,0.30,1,129.99,129.99,39.00,South America,COMPLETE,403,18,Nike Men's CJ Elite 2 TD Football Cleat,129.99,Standard Class,03-21-2017,03-18-2017,04:39
57488,DEBIT,3,4,60.98,168.00,Advance shipping,0,24,Women's Apparel,Caguas,Puerto Rico,Joshua,6255,Smith,Consumer,PR,2225 Quiet Woods,725,5,Golf,18.228205,-66.370499,San Salvador,El Salvador,6255,52312,502,32.00,0.16,130719,50.00,0.36,4,200.00,168.00,60.98,Central America,ON_HOLD,502,24,Nike Men's Dri-FIT Victory Golf Polo,50.00,Standard Class,02-05-2017,02-02-2017,14:52
126482,PAYMENT,4,4,-101.24,337.46,Shipping on time,0,63,Children's Clothing,Caguas,Puerto Rico,Dora,13891,Justice,Consumer,PR,1565 Silver Place,725,4,Apparel,18.221752,-66.037064,Casoria,Italia,13891,70338,1350,19.64,0.06,173653,357.10,-0.30,1,357.10,337.46,-101.24,Southern Europe,PENDING_PAYMENT,1350,63,Children's heaters,357.10,Standard Class,10-27-2017,10-23-2017,18:11
16471,CASH,5,2,47.88,251.98,Late delivery,1,43,Camping & Hiking,San Francisco,EE. UU.,Mary,7721,Jenkins,Consumer,CA,3884 Golden Landing,94122,7,Fan Shop,37.761757,-122.471352,Wollongong,Australia,7721,24864,957,48.00,0.16,62290,299.98,0.19,1,299.98,251.98,47.88,Oceania,CLOSED,957,43,Diamondback Women's Serene Classic Comfort Bi,299.98,Second Class,01-03-2016,12-29-2015,22:36
128236,DEBIT,4,4,-37.38,51.00,Shipping on time,0,26,Girls' Apparel,San Jose,EE. UU.,Lori,3246,Burns,Consumer,CA,4845 Thunder Embers Hollow,95148,5,Golf,37.325542,-121.799347,Edmonds,Estados Unidos,3246,36753,564,9.00,0.15,91715,30.00,-0.73,2,60.00,51.00,-37.38,West of USA,COMPLETE,564,26,Nike Men's Deutschland Weltmeister Winners Bl,30.00,Standard Class,06-24-2016,06-20-2016,11:51


In [56]:
dim_products = df_data[['product_card_id', 'product_name', 'product_price', 
                        'category_id', 'category_name', 'department_id', 
                        'department_name']].drop_duplicates(subset=['product_card_id']).reset_index(drop=True)

In [57]:
dim_products.tail(5)

,product_card_id,product_name,product_price,category_id,category_name,department_id,department_name
113,646,Merrell Women's Grassbow Sport Hiking Shoe,99.99,30,Men's Golf Clubs,6,Outdoors
114,1361,Toys,11.54,74,Toys,7,Fan Shop
115,1073,Pelican Sunstream 100 Kayak,199.99,48,Water Sports,7,Fan Shop
116,1059,Pelican Maverick 100X Kayak,349.99,48,Water Sports,7,Fan Shop
117,1014,O'Brien Men's Neoprene Life Vest,49.98,46,Indoor/Outdoor Games,7,Fan Shop


In [58]:
dim_customers = df_data[['customer_id', 'customer_fname', 'customer_lname', 
                    'customer_segment', 'customer_street', 
                    'customer_city', 'customer_state', 'customer_zipcode', 'customer_country'
                    ]].drop_duplicates(subset=['customer_id']).reset_index(drop=True)

In [59]:
dim_customers.head(5)

,customer_id,customer_fname,customer_lname,customer_segment,customer_street,customer_city,customer_state,customer_zipcode,customer_country
0,20755,Cally,Holloway,Consumer,5365 Noble Nectar Island,Caguas,PR,725,Puerto Rico
1,19492,Irene,Luna,Consumer,2679 Rustic Loop,Caguas,PR,725,Puerto Rico
2,19491,Gillian,Maldonado,Consumer,8510 Round Bear Gate,San Jose,CA,95125,EE. UU.
3,19490,Tana,Tate,Home Office,3200 Amber Bend,Los Angeles,CA,90027,EE. UU.
4,19489,Orli,Hendricks,Corporate,8671 Iron Anchor Corners,Caguas,PR,725,Puerto Rico


In [60]:
dim_stores = df_data[['latitude', 'longitude', 'shipping_mode']].drop_duplicates().reset_index(drop=True)
dim_stores['store_id'] = dim_stores.index + 1
dim_stores = dim_stores[['store_id', 'latitude', 'longitude', 'shipping_mode']]

In [61]:
dim_stores.sample(5)

,store_id,latitude,longitude,shipping_mode
8874,8875,47.208981,-122.211838,First Class
11754,11755,39.453449,-77.560112,Second Class
4972,4973,37.726482,-121.434944,Standard Class
17320,17321,40.063980,-80.721413,Second Class
15776,15777,47.926037,-122.417801,First Class


In [62]:
fact_sales = pd.merge(df_data, dim_stores, on=['latitude', 'longitude', 'shipping_mode'], how='left')

In [63]:
fact_sales.sample(5)

,type,days_for_shipping_real,days_for_shipment_scheduled,benefit_per_order,sales_per_customer,delivery_status,late_delivery_risk,category_id,category_name,customer_city,customer_country,customer_fname,customer_id,customer_lname,customer_segment,customer_state,customer_street,customer_zipcode,department_id,department_name,latitude,longitude,order_city,order_country,order_customer_id,order_id,order_item_cardprod_id,order_item_discount,order_item_discount_rate,order_item_id,order_item_product_price,order_item_profit_ratio,order_item_quantity,sales,order_item_total,order_profit_per_order,order_region,order_status,product_card_id,product_category_id,product_name,product_price,shipping_mode,delivery_date,order_date,order_time,store_id
109618,TRANSFER,2,1,44.59,122.84,Late delivery,1,18,Men's Footwear,Los Angeles,EE. UU.,Mary,1570,Smith,Consumer,CA,8544 Silent Treasure Thicket,90057,4,Apparel,34.061081,-118.277473,Hurghada,Egipto,1570,42095,403,7.15,0.06,105098,129.99,0.36,1,129.99,122.84,44.59,North Africa,PENDING,403,18,Nike Men's CJ Elite 2 TD Football Cleat,129.99,First Class,09-08-2016,09-06-2016,11:24,3627
136365,TRANSFER,5,4,15.30,152.97,Late delivery,1,17,Cleats,Chicago,EE. UU.,Mary,12065,Hogan,Consumer,IL,2514 Wishing Rabbit Grounds,60628,4,Apparel,41.692806,-87.617157,Limoeiro do Norte,Brasil,12065,10144,365,27.00,0.15,25341,59.99,0.10,3,179.97,152.97,15.30,South America,PROCESSING,365,17,Perfect Fitness Perfect Rip Deck,59.99,Standard Class,06-03-2015,05-29-2015,01:32,3064
160830,TRANSFER,4,4,-5.12,20.49,Shipping on time,0,5,Lacrosse,Caguas,Puerto Rico,Thomas,8869,Lambert,Consumer,PR,2777 Crystal Blossom Dale,725,2,Fitness,18.259886,-66.370613,San Francisco,Estados Unidos,8869,38803,93,4.50,0.18,96888,24.99,-0.25,1,24.99,20.49,-5.12,West of USA,PROCESSING,93,5,Under Armour Men's Tech II T-Shirt,24.99,Standard Class,07-24-2016,07-20-2016,10:04,14130
33986,CASH,5,4,66.59,183.45,Late delivery,1,76,Women's Clothing,Holland,EE. UU.,Kimberly,20452,White,Consumer,MI,2088 Emerald Downs,49423,4,Apparel,42.831871,-86.078522,Cainta,Filipinas,20452,76899,1363,32.37,0.15,180214,215.82,0.36,1,215.82,183.45,66.59,Southeast Asia,CLOSED,1363,76,Summer dresses,215.82,Standard Class,02-01-2018,01-27-2018,12:47,3268
121511,TRANSFER,4,4,5.37,47.50,Shipping on time,0,24,Women's Apparel,Lancaster,EE. UU.,Heather,6587,Henson,Consumer,PA,6724 Crystal Dale,17602,5,Golf,40.042091,-76.299110,Las Tunas,Cuba,6587,58788,502,2.50,0.05,147153,50.00,0.11,1,50.00,47.50,5.37,Caribbean,PENDING,502,24,Nike Men's Dri-FIT Victory Golf Polo,50.00,Standard Class,05-12-2017,05-08-2017,03:42,2815


In [65]:
fact_sales = fact_sales[[
    'order_item_id', 'order_id', 'customer_id', 'product_card_id', 'store_id',
    'order_date', 'order_time',
    'type', 'order_status', 'days_for_shipping_real', 
    'days_for_shipment_scheduled', 'delivery_status', 'late_delivery_risk',
    'order_city', 'order_country', 'order_region',
    'sales', 'order_item_quantity', 'order_item_product_price', 
    'order_item_discount', 'order_item_discount_rate', 'order_item_total', 
    'order_item_profit_ratio', 'benefit_per_order', 'order_profit_per_order', 'sales_per_customer'
]]
 
print("¡DataFrames del Modelo en Estrella listos en memoria!")

¡DataFrames del Modelo en Estrella listos en memoria!


In [66]:
USER  = os.getenv("DB_USER")
PASS  = os.getenv("DB_PASSWORD")
HOST  = os.getenv("DB_HOST")
PORT  = os.getenv("DB_PORT", "3306")   # 3306 como valor por defecto
DB_NAME = os.getenv("DB_NAME")
 
# 1️⃣ Conectar al servidor SIN base de datos

engine_server = create_engine(f"mysql+pymysql://{USER}:{PASS}@{HOST}:{PORT}") 
# 2️⃣ Crear la base de datos si no existe

with engine_server.begin() as con:
    con.execute(text(f"CREATE DATABASE IF NOT EXISTS {DB_NAME}"))
    print(f"✅ Base de datos '{DB_NAME}' lista")
 
engine_server.dispose()
 
# 3️⃣ Conectar ahora CON la base de datos

engine = create_engine(f"mysql+pymysql://{USER}:{PASS}@{HOST}:{PORT}/{DB_NAME}")
print(f"✅ Conectado a '{DB_NAME}'")
 

✅ Base de datos 'dataco' lista
✅ Conectado a 'dataco'


In [67]:
# Carga del DataFrame en MySQL
# Esto crea automáticamente la tabla products en la base de datos
 
dim_products.to_sql("products", con=engine, if_exists="replace", index=False)
 
'''# Comprobación previa para evitar errores al asignar la clave primaria
dim_products = dim_products.dropna(subset=["Employee_Number"])
dim_products = dim_products.drop_duplicates(subset=["Employee_Number"])'''
 
dim_products.to_sql("products", con=engine, if_exists="replace", index=False)
 
# Asignación de la clave primaria una vez creada la tabla
with engine.begin() as con:
    con.execute(text("""
        ALTER TABLE products
        MODIFY product_card_id INT NOT NULL"""))
    con.execute(text("""
        ALTER TABLE products
        ADD PRIMARY KEY (product_card_id)"""))
 
print("¡Listo! Base de datos, tabla y clave primaria creadas correctamente.")

¡Listo! Base de datos, tabla y clave primaria creadas correctamente.
